In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

## Load data files & Mouse trajectory preview

In [ ]:
from src.plotting import plot_trajectory
from src.data import find_red_eclipse_files, load_red_eclipse_mouse

game_files = find_red_eclipse_files()

for file_path in game_files[:5]:
    meta, mouse = load_red_eclipse_mouse(file_path)
    if mouse.empty:
        continue
    plot_trajectory(mouse, title=f"userId={meta['userId']}, gameId={meta['gameId']}")
    print(f"\nuserId={meta['userId']}, gameId={meta['gameId']}, events={len(mouse)}")


## Feature extraction


In [ ]:
from src.artifacts_store import (
    apply_bot_cache,
    save_game_cache,
    train_or_load_bot_detector,
    try_load_game_cache,
)

N_PREVIEW = 5


def save_human_or_bot_cache(game, split, features_df, traces, session_ids, bot_types=None, user_ids=None):
    save_game_cache(
        game=game,
        split=split,
        features_df=features_df,
        traces=traces,
        session_ids=session_ids,
        bot_types=bot_types,
        user_ids=user_ids,
    )


In [ ]:
from src.features import extract_features
from src.data import load_red_eclipse_mouse

re_cache = try_load_game_cache("re", "human")
if re_cache is not None:
    re_games_df = re_cache["features"]
    re_human_traces = re_cache["traces"]
    print(f"Loaded RE human cache: features={len(re_games_df)} traces={len(re_human_traces)}")
else:
    re_rows = []
    re_human_traces = []

    for file_path in game_files:
        re_meta, re_mouse = load_red_eclipse_mouse(file_path)
        features = extract_features(re_mouse)

        if features is None:
            continue

        features.update({
            **re_meta,
            "is_bot": 0,
            "bot_type": "human",
        })
        re_rows.append(features)
        re_human_traces.append(re_mouse)

    re_games_df = pd.DataFrame(re_rows)
    save_human_or_bot_cache(
        game="re",
        split="human",
        features_df=re_games_df,
        traces=re_human_traces,
        session_ids=re_games_df["gameId"].astype(str).tolist(),
        bot_types=["human"] * len(re_human_traces),
        user_ids=re_games_df["userId"].astype(str).tolist(),
    )
    print(f"Saved RE human cache: features={len(re_games_df)} traces={len(re_human_traces)}")

print(re_games_df.head())


## Player identification


In [ ]:
from src.features import feature_cols

select_model = re_games_df

input_data = select_model[feature_cols]
output_data = select_model["userId"]

input_train, input_test, output_train, output_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42, stratify=output_data
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(input_train, output_train)

output_pred = model.predict(input_test)
accuracy = accuracy_score(output_test, output_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"Random baseline: {1 / output_data.nunique():.2%} ({output_data.nunique()} players, {len(select_model)} games)")
print()
print(classification_report(output_test, output_pred))


## Block-bootstrap synthetic bots


In [ ]:
from src.bots import (
    build_segments,
    stitch_bot_game,
    collect_human_motion_samples,
    median_trace_duration_ms,
)
from src.features import extract_features
from src.config import RNG_SEED
from src.data import load_red_eclipse_mouse

rng = np.random.default_rng(RNG_SEED)

re_segment_pool = []
for file_path in game_files:
    re_meta, re_mouse = load_red_eclipse_mouse(file_path)
    re_segment_pool.extend(build_segments(re_mouse, rng=rng))

re_motion = collect_human_motion_samples(re_human_traces, rng=rng)
re_target_ms = median_trace_duration_ms(re_human_traces)
print(
    f"Segment pool: {len(re_segment_pool)} segments from {len(game_files)} games | "
    f"dt n={len(re_motion['dt_samples'])} sessions={len(re_motion['dt_by_session'])} median={np.median(re_motion['dt_samples']):.2f}ms | "
    f"step median={np.median(re_motion['step_samples']):.3f} | "
    f"target={re_target_ms/1000:.1f}s"
)

re_bot_cache = try_load_game_cache('re', 'bot')
if re_bot_cache is not None:
    apply_bot_cache('re', re_bot_cache, globals(), n_preview=N_PREVIEW)
    RE_BOTS_READY = True
    print(f"Loaded RE bot cache: features={len(re_bot_cache['features'])} traces={len(re_bot_cache['traces'])}")
else:
    RE_BOTS_READY = False
    re_stitch_rows = []
    re_stitch_traces = []
    sample_re_stitch_trajectories = []
    for i in range(len(re_games_df)):
        bot_mouse = stitch_bot_game(
            re_segment_pool,
            dt_samples=re_motion['dt_samples'],
            dt_by_session=re_motion['dt_by_session'],
            target_duration_ms=re_target_ms,
            rng=rng,
        )
        re_stitch_traces.append(bot_mouse)
        if len(sample_re_stitch_trajectories) < N_PREVIEW:
            sample_re_stitch_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({
            'userId': -1,
            'gameId': f'bot_{i}',
            'source_file': f'synthetic_bot_{i}',
            'is_bot': 1,
            'bot_type': 'stitch',
        })
        re_stitch_rows.append(feats)
    re_stitch_df = pd.DataFrame(re_stitch_rows)
    print(f"Generated {len(re_stitch_df)} stitch bot games")
    print(re_stitch_df.head())


## Block-bootstrap synthetic bots trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_re_stitch_trajectories):
    plot_trajectory(df, title=f"bot_{i}")
    print(f"\nbot_{i}, events={len(df)}")


## Smooth bot generation


In [ ]:
from src.bots import (
    estimate_smooth_params,
    generate_smooth_bot_game,
    smooth_generator_params,
    smooth_params_for_print,
)
from src.features import extract_features

re_median_events = int(re_games_df['n_events'].median())
re_smooth_params = estimate_smooth_params(re_games_df, **re_motion)
print(f"RE smooth params: {smooth_params_for_print(re_smooth_params)}")

if not RE_BOTS_READY:
    re_smooth_gen = smooth_generator_params(re_smooth_params)
    re_smooth_rows = []
    re_smooth_traces = []
    sample_re_smooth_trajectories = []
    for i in range(len(re_games_df)):
        bot_mouse = generate_smooth_bot_game(
            n_events=max(re_median_events * 3, 1),
            seed=RNG_SEED + i,
            target_duration_ms=re_target_ms,
            **re_smooth_gen,
        )
        re_smooth_traces.append(bot_mouse)
        if len(sample_re_smooth_trajectories) < N_PREVIEW:
            sample_re_smooth_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({
            'userId': -2,
            'gameId': f'smooth_{i}',
            'source_file': f'synthetic_smooth_{i}',
            'is_bot': 1,
            'bot_type': 'smooth',
        })
        re_smooth_rows.append(feats)
    re_smooth_df = pd.DataFrame(re_smooth_rows)
    print(f"Generated {len(re_smooth_df)} smooth bot games (n_events={re_median_events})")
    print(re_smooth_df.head())
else:
    print('RE smooth bots loaded from cache')


## Smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_re_smooth_trajectories):
    plot_trajectory(df, title=f"smooth_{i}")
    print(f"\nsmooth_{i}, events={len(df)}")

## Bézier bot generation


In [ ]:
from src.bots import (
    estimate_bezier_params,
    generate_bezier_bot_game,
    bezier_params_for_print,
)
from src.features import extract_features

N_BEZIER_BOTS = len(re_games_df)
re_bezier_params = estimate_bezier_params(re_games_df, **re_motion)
print(f"RE bezier params: {bezier_params_for_print(re_bezier_params)}")

if not RE_BOTS_READY:
    bezier_rows = []
    re_bezier_traces = []
    sample_re_bezier_trajectories = []
    for i in range(N_BEZIER_BOTS):
        bot_mouse = generate_bezier_bot_game(
            n_events=max(re_median_events * 3, 1),
            seed=RNG_SEED + 50 + i,
            target_duration_ms=re_target_ms,
            **re_bezier_params,
        )
        re_bezier_traces.append(bot_mouse)
        if len(sample_re_bezier_trajectories) < N_PREVIEW:
            sample_re_bezier_trajectories.append(bot_mouse.copy())
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({
            'userId': -3,
            'gameId': f'bezier_{i}',
            'source_file': f'synthetic_bezier_{i}',
            'is_bot': 1,
            'bot_type': 'bezier',
        })
        bezier_rows.append(feats)
    re_bezier_df = pd.DataFrame(bezier_rows)
    print(f"Generated {len(re_bezier_df)} bezier bot games (n_events={re_median_events})")
    print(re_bezier_df.head())
else:
    print('RE bezier bots loaded from cache')


## Bézier bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_re_bezier_trajectories):
    plot_trajectory(df, title=f"bezier_{i}")
    print(f"\nbezier_{i}, events={len(df)}")



## VAE bot — train once / load weights (RE)


In [ ]:
from pathlib import Path
import numpy as np

from src.config import RNG_SEED
from src.vae_bot import ensure_vae_bundle, DEFAULT_RE_WEIGHTS

# Formal runs: False → load committed weights. True → retrain + overwrite path.
VAE_FORCE_RETRAIN = False
VAE_WEIGHTS_PATH = DEFAULT_RE_WEIGHTS  # artifacts/vae_re_v2.pt (per-axis std)

re_step_median = float(np.median(re_motion["step_samples"]))
re_vae_bundle = ensure_vae_bundle(
    re_human_traces,
    re_step_median,
    path=VAE_WEIGHTS_PATH,
    force_retrain=VAE_FORCE_RETRAIN,
    seed=RNG_SEED,
)
print(
    f"RE VAE ready | path={Path(VAE_WEIGHTS_PATH)} | "
    f"seg_len={re_vae_bundle['seg_len']} z={re_vae_bundle['z_dim']} "
    f"norm={re_vae_bundle.get('norm')} axis_scale={re_vae_bundle.get('axis_scale')} "
    f"trained_segments={re_vae_bundle.get('n_segments')}"
)


## VAE bot generation (RE)


In [ ]:
from src.vae_bot import generate_vae_bot_games
from src.features import extract_features
from src.config import RNG_SEED, VAE_POOL_SEGMENTS

if not RE_BOTS_READY:
    re_vae_rng = np.random.default_rng(RNG_SEED + 3)
    re_vae_traces = generate_vae_bot_games(
        re_vae_bundle,
        n_games=len(re_games_df),
        dt_samples=re_motion['dt_samples'],
        dt_by_session=re_motion['dt_by_session'],
        target_duration_ms=re_target_ms,
        n_pool_segments=VAE_POOL_SEGMENTS,
        rng=re_vae_rng,
    )
    sample_vae_trajectories = [m.copy() for m in re_vae_traces[:N_PREVIEW]]
    re_vae_rows = []
    for i, bot_mouse in enumerate(re_vae_traces):
        feats = extract_features(bot_mouse)
        if feats is None:
            continue
        feats.update({
            'userId': -4,
            'gameId': f'vae_{i}',
            'source_file': f'synthetic_vae_{i}',
            'is_bot': 1,
            'bot_type': 'vae',
        })
        re_vae_rows.append(feats)
    re_vae_df = pd.DataFrame(re_vae_rows)
    print(f"Generated {len(re_vae_df)} VAE bot games (pool={VAE_POOL_SEGMENTS} segs/game)")
    print(re_vae_df.head())
else:
    print('RE vae bots loaded from cache')


In [ ]:
if not globals().get('RE_BOTS_READY'):
    re_bot_features = pd.concat(
        [re_stitch_df, re_smooth_df, re_bezier_df, re_vae_df], ignore_index=True
    )
    re_bot_traces = re_stitch_traces + re_smooth_traces + re_bezier_traces + re_vae_traces
    save_human_or_bot_cache(
        game='re',
        split='bot',
        features_df=re_bot_features,
        traces=re_bot_traces,
        session_ids=re_bot_features['gameId'].astype(str).tolist(),
        bot_types=re_bot_features['bot_type'].astype(str).tolist(),
        user_ids=re_bot_features['userId'].astype(str).tolist(),
    )
    RE_BOTS_READY = True
    print(f"Saved RE bot cache: features={len(re_bot_features)} traces={len(re_bot_traces)}")


## VAE bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_vae_trajectories):
    plot_trajectory(df, title=f"vae_{i}")
    print(f"\nvae_{i}, events={len(df)}")


## RE Human vs Bot classification (GroupKFold + split-player + windows)

In [ ]:
from src.evaluation import evaluate_group_kfold_windows
from src.features import feature_cols
from src.data import load_red_eclipse_mouse
from pathlib import Path

re_file_by_name = {Path(p).name: p for p in game_files}
# read the mouse data from the source file
def _re_traces_for(feat_df):
    return [load_red_eclipse_mouse(re_file_by_name[src])[1] for src in feat_df["source_file"]]

re_cross_result = evaluate_group_kfold_windows(
    human_df=re_games_df,
    groups=re_games_df["userId"].to_numpy(),
    traces_for_df=_re_traces_for,
    feature_cols=feature_cols,
    vae_bundle=re_vae_bundle,
    name="RE",
)
print("\nRE fold table (window counts):")
print(
    re_cross_result["fold_df"][
        [
            "fold",
            "n_train_sessions",
            "n_test_sessions",
            "n_train_windows_human",
            "n_test_windows_human",
            "n_test_groups",
        ]
    ].to_string(index=False)
)
print("\nRE window-level summary:")
print(re_cross_result["summary_df"].to_string(index=False))
if re_cross_result["session_summary_df"] is not None:
    print("\nRE session-mean-of-windows summary:")
    print(re_cross_result["session_summary_df"].to_string(index=False))


## RE train Cross-game transfer features

In [ ]:
from src.features import cross_game_feature_cols, to_scale_invariant, SCALE_INVARIANT_COLS, SCALE_INVARIANT_EXT_COLS
from src.config import RNG_SEED

print("=== RE model trained on STITCH bots ===")
re_model_stitch, re_acc_stitch = train_or_load_bot_detector(
    re_games_df, re_stitch_df, cross_game_feature_cols,
    train_game="re", feature_set="raw", bot_type="stitch",
    name="stitch",
)
print()
print("=== RE model trained on SMOOTH bots ===")
re_model_smooth, re_acc_smooth = train_or_load_bot_detector(
    re_games_df, re_smooth_df, cross_game_feature_cols,
    train_game="re", feature_set="raw", bot_type="smooth",
    name="smooth",
)
print()
print("=== RE model trained on BEZIER bots ===")
re_model_bezier, re_acc_bezier = train_or_load_bot_detector(
    re_games_df, re_bezier_df, cross_game_feature_cols,
    train_game="re", feature_set="raw", bot_type="bezier",
    name="bezier",
)
print()
print("=== RE model trained on VAE bots ===")
re_model_vae, re_acc_vae = train_or_load_bot_detector(
    re_games_df, re_vae_df, cross_game_feature_cols,
    train_game="re", feature_set="raw", bot_type="vae",
    name="vae",
)

# calculate scale invariant features
re_si_human = to_scale_invariant(re_games_df)
re_si_stitch = to_scale_invariant(re_stitch_df)
re_si_smooth = to_scale_invariant(re_smooth_df)
re_si_bezier = to_scale_invariant(re_bezier_df)
re_si_vae = to_scale_invariant(re_vae_df)

# train scale invariant(4) features
print("=== Train on Red Eclipse (scale-invariant features) ===")
m_si_stitch, _ = train_or_load_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_COLS,
    train_game="re", feature_set="si_min", bot_type="stitch",
    random_state=RNG_SEED, name="scale-invariant stitch",
    show_feature_importance=False,
)
print()
m_si_smooth, _ = train_or_load_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_COLS,
    train_game="re", feature_set="si_min", bot_type="smooth",
    random_state=RNG_SEED, name="scale-invariant smooth",
    show_feature_importance=False,
)
print()
m_si_bezier, _ = train_or_load_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_COLS,
    train_game="re", feature_set="si_min", bot_type="bezier",
    random_state=RNG_SEED, name="scale-invariant bezier",
    show_feature_importance=False,
)

print()
m_si_vae, _ = train_or_load_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_COLS,
    train_game="re", feature_set="si_min", bot_type="vae",
    random_state=RNG_SEED, name="scale-invariant vae",
    show_feature_importance=False,
)

# train scale invariant(10) features
print("=== Train on Red Eclipse (scale-invariant EXT, 10 feats) ===")
print("cols:", SCALE_INVARIANT_EXT_COLS)
m_si_ext_stitch, _ = train_or_load_bot_detector(
    re_si_human, re_si_stitch, SCALE_INVARIANT_EXT_COLS,
    train_game="re", feature_set="si_ext", bot_type="stitch",
    random_state=RNG_SEED, name="SI-EXT stitch",
    show_feature_importance=False,
)
print()
m_si_ext_smooth, _ = train_or_load_bot_detector(
    re_si_human, re_si_smooth, SCALE_INVARIANT_EXT_COLS,
    train_game="re", feature_set="si_ext", bot_type="smooth",
    random_state=RNG_SEED, name="SI-EXT smooth",
    show_feature_importance=False,
)
print()
m_si_ext_bezier, _ = train_or_load_bot_detector(
    re_si_human, re_si_bezier, SCALE_INVARIANT_EXT_COLS,
    train_game="re", feature_set="si_ext", bot_type="bezier",
    random_state=RNG_SEED, name="SI-EXT bezier",
    show_feature_importance=False,
)
print()
m_si_ext_vae, _ = train_or_load_bot_detector(
    re_si_human, re_si_vae, SCALE_INVARIANT_EXT_COLS,
    train_game="re", feature_set="si_ext", bot_type="vae",
    random_state=RNG_SEED, name="SI-EXT vae",
    show_feature_importance=False,
)